# ML-KEM B: poly_tomsg 트레이스 수집

A 수집 노트북(`mlkem-trace-basemul.ipynb`)과 **동일한 캡처 절차**를 쓴다. `TRIGGER_A=0`, `TRIGGER_B=1`로 B 함수만 트리거하고, B에 맞게 `SAMPLES`를 줄인다. 키와 암호문은 이 노트북에서 새로 생성하므로, A 파일과 행별로 대응되는 수집은 아니다.

## 1. 설정

In [3]:
SCOPETYPE = "OPENADC"          # CW-Lite / CW-Pro / Husky
PLATFORM  = "CW308_STM32F4"    # CW308T-STM32F4HWC (STM32F415)도 동일
SS_VER    = "SS_VER_2_1"         # 펌웨어는 SimpleSerial 2.1만 지원 (Setup에서 target = SimpleSerial2)

PARAMS     = 768               # 512 | 768 | 1024  (ML-KEM 스킴)
IMPL       = "clean"           # "m4fstack" | "m4fspeed" | "clean" — 트리거(A/B)는 clean에만 있음
TRIGGER_A  = 0                 # A 트리거 끔: B 시작 상승 에지 하나만 캡처
TRIGGER_B  = 1                 # B(poly_tomsg) 호출 앞뒤에서 트리거
OPT        = "3"               # 컴파일 최적화: "0" = -O0 (feasibility, 누출 명확) / "3" = -O3 (pqm4 기본)
N_TRACES   = 1000              # 수집할 B 파형 수 (고정 키, 매번 새 암호문)
ADC_MUL    = 2                 # 타깃 클럭 대비 ADC 배수; 키 로드 전에 설정하고 타깃 리셋
SAMPLES    = 20000             # B 함수만 캡처; 실측 trig_count > SAMPLES면 중단하고 늘리기


In [4]:
%run "Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.adc.trig_count                     changed from 286668                    to 15223764                 
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx         

In [5]:
%%bash -s "$PLATFORM" "$SS_VER" "$PARAMS" "$IMPL" "$TRIGGER_A" "$TRIGGER_B" "$OPT"
cd ../firmware/mcu/simpleserial-mlkem                                   # 펌웨어 빌드 디렉토리로 이동
make PLATFORM=$1 SS_VER=$2 PARAMS=$3 IMPL=$4 TRIGGER_A=$5 TRIGGER_B=$6 OPT=$7 -j   # 설정대로 컴파일(-j 병렬)

SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:13.2.rel1-2) 13.2.1 20231009
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CW308_STM32F4 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
.
.
Compiling:
.
.
Compiling:
-en     simpleserial-mlkem.c ...
Compiling:
Compiling:
Compiling:
-en     .././pqm4/mupq/pqclean/crypto_kem/ml-kem-768/clean/cbd.c ...
Compiling:
-en     .././pqm4/mupq/pqclean/crypto_kem/ml-kem-768/clean/kem.c ...
-en     .././pqm4/mupq/pqclean/crypto_kem/ml-kem-768/clean/indcpa.c ...
-en     .././pqm4/mupq/pqclean/crypto_kem/ml-kem-768/clean/ntt.c ...
-en     .././pqm4/mupq/pqclean/crypto_kem/ml-kem-768/clean/polyvec.c ...
-en     .././pqm4/mupq/pqclean/crypto_kem/ml-kem-768/clean/poly.c ...
.
.
.
Compiling:
.
.
.
.
.
.
.
.
.
Compiling:
Compiling:
Compiling:
-en     .././pqm4/

In file included from .././hal/chipwhisperer-fw-extra/stm32f4/stm32f4_hal_lowlevel.c:39:
.././hal/chipwhisperer-fw-extra/stm32f4/stm32f4_hal_lowlevel.h:108: warning: "STM32F415xx" redefined
  108 | #define STM32F415xx
      | 
<command-line>: note: this is the location of the previous definition
In file included from .././hal/chipwhisperer-fw-extra/stm32f4/stm32f4_hal.c:3:
.././hal/chipwhisperer-fw-extra/stm32f4/stm32f4_hal_lowlevel.h:108: warning: "STM32F415xx" redefined
  108 | #define STM32F415xx
      | 
<command-line>: note: this is the location of the previous definition


-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     simpleserial-mlkem768-clean-CW308_STM32F4.elf ...


/usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-closer.o): in function `_close_r':
/build/newlib-38V0JC/newlib-4.4.0.20231231/build/arm-none-eabi/thumb/v7e-m/nofp/newlib/../../../../../../newlib/libc/reent/closer.c:47:(.text+0xc): warning: _close is not implemented and will always fail
/usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-lseekr.o): in function `_lseek_r':
/build/newlib-38V0JC/newlib-4.4.0.20231231/build/arm-none-eabi/thumb/v7e-m/nofp/newlib/../../../../../../newlib/libc/reent/lseekr.c:49:(.text+0x14): warning: _lseek is not implemented and will always fail
/usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-readr.o): in function `_read_r':
/build/newlib-38V

/usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-closer.o): note: the message above does not take linker garbage collection into account
/usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-lseekr.o): note: the message above does not take linker garbage collection into account
/usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-readr.o): note: the message above does not take linker garbage collection into account
/usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/bin/ld: /usr/lib/gcc/arm-none-eabi/13.2.1/../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-writer.o): note: the message above does not take linker garbage collection into account
-e Done!
.
.
.
.
.
Cr

In [6]:
fw_path = "../firmware/mcu/simpleserial-mlkem/simpleserial-mlkem{}-{}-{}.hex".format(  # 빌드된 hex 경로
    PARAMS, IMPL, PLATFORM)                      # 파일명에 파라미터/구현/플랫폼이 들어감
cw.program_target(scope, prog, fw_path)          # STM32에 펌웨어 플래시
reset_target(scope)                              # 보드 리셋(nRST): RAM 초기화 후 펌웨어 재시작

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 18547 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 18547 bytes


In [7]:
# ============================================================
# 4. 기능 검증
#   - 보드에 올린 펌웨어가 설정(PARAMS/IMPL/TRIGGER_A/B)과 같은지 확인
#   - 보드 decaps 결과를 PC의 kyber-py(기준 구현)와 비교
# ============================================================
from kyber_py.ml_kem import ML_KEM_512, ML_KEM_768, ML_KEM_1024   # PC 기준 구현(키/암호문/정답 생성용)

# PARAMS에 맞는 KEM 객체 선택 (키 생성·암호문 생성·정답 공유비밀 계산은 전부 PC가 함)
KEM = {512: ML_KEM_512, 768: ML_KEM_768, 1024: ML_KEM_1024}[PARAMS]

# ── ADC 샘플링 배수 설정 (반드시 여기서! = 키 로드·통신 '전') ──────────────
# scope.clock.adc_mul 변경은 Husky PLL을 재설정 → 타깃 클럭(clkgen)이 순간 끊겨 보드가 통신 불능이 됨
#   ("Target clock may drop; reset your target" 경고) → 바꾼 직후 타깃을 리셋해 재부팅시켜야 UART 복구됨.
# 수집 셀(키 로드 후)에서 바꾸면 로드된 키·통신이 깨지므로, 반드시 이 지점에서 1회만 바꾼다.
import time                                          # 리셋 대기용
scope.clock.adc_mul = ADC_MUL                        # 키 적재 전에 ADC PLL 설정 (B 캡처도 동일)
time.sleep(0.3)                                       # PLL 재잠금 대기
scope.io.nrst = "low";   time.sleep(0.05)             # 타깃 리셋 펄스 (클럭 흔들림 후 보드 재부팅)
scope.io.nrst = "high_z"; time.sleep(0.5)             # 리셋 해제(풀업) + 부팅 대기
target.flush()                                        # 리셋 중 쌓인 UART 쓰레기 비우기
print(f"adc_mul={scope.clock.adc_mul}  adc_freq={scope.clock.adc_freq/1e6:.2f} MHz  (clkgen 7.37MHz × adc_mul, target reset done)")

# ------------------------------------------------------------
# 펌웨어 정보 확인 ('i' -> 'r' 4B)
#   fw[0] = PARAMS/256   (2=512, 3=768, 4=1024)
#   fw[1] = 구현         (1=m4fstack, 2=m4fspeed, 3=clean)
#   fw[2] = 트리거 비트   (bit0=A: 비밀키 basemul, bit1=B: poly_tomsg)
#   fw[3] = 조각 크기     (240)
# 트리거 비트는 "컴파일할 때 매크로가 켜졌는지"만 알려줌
#   -> 실제로 펄스가 나는지는 파형/에뮬레이터로 따로 확인
# simpleserial_read는 'r' 다음에 오는 'e'(ack) 패킷까지 읽어서 버림
# ------------------------------------------------------------
target.simpleserial_write("i", bytearray())      # 펌웨어에 정보 요청 명령 'i' 전송
fw = target.simpleserial_read("r", 4, timeout=1000)   # 응답 'r' 4바이트 수신
if fw is None:                                   # 응답 없으면
    raise RuntimeError("'i' 응답 없음: 보드 리셋/통신/SS_VER 확인")   # 통신 문제로 중단

impl_name = {1: "m4fstack", 2: "m4fspeed", 3: "clean"}   # fw[1] 숫자 -> 구현 이름
fw_params, fw_impl, fw_trigger = fw[0] * 256, impl_name.get(fw[1]), fw[2]   # 응답 해석
print("firmware:", fw_params, fw_impl, f"trigger=0b{fw_trigger:02b} (bit0=A bit1=B)", "chunk", fw[3])  # 표시

# 설정 셀 값으로 기대 트리거 비트 생성 -> 엉뚱한 hex를 플래시한 채로 수집하는 실수 방지
want_trigger = (1 if TRIGGER_A else 0) | (2 if TRIGGER_B else 0)   # bit0=A, bit1=B
assert (fw_params, fw_impl, fw_trigger) == (PARAMS, IMPL, want_trigger), "플래시된 펌웨어가 설정과 다릅니다"  # 불일치면 중단

# SimpleSerial 2.1 패킷 하나에 데이터는 최대 249B라 dk(768: 2400B), ct(768: 1088B)를 조각으로 보냄
CHUNK = fw[3]                                     # 조각 크기는 펌웨어가 알려준 값(240)을 그대로 사용


def send_chunks(cmd, data):
    """data를 CHUNK 바이트씩 잘라 보냄: scmd = 조각 번호, 마지막 조각은 남은 길이 그대로 (패딩 없음)"""
    for idx, i in enumerate(range(0, len(data), CHUNK)):   # CHUNK 단위로 끊으며 조각 번호 idx
        target.send_cmd(cmd, idx, data[i:i + CHUNK])       # 펌웨어가 (idx*CHUNK) 위치에 memcpy
        err = target.simpleserial_wait_ack(timeout=500)    # 조각마다 'e' ack 확인
        if err is None:                                    # 응답 없으면
            raise RuntimeError(f"'{cmd}' 조각 {idx}: ack 없음")
        if err[0] != 0:                                    # 0이 아니면 펌웨어 에러 코드
            raise RuntimeError(f"'{cmd}' 조각 {idx}: 펌웨어 에러 0x{err[0]:02x}")


def load_key(dk):
    """비밀키(dk, FIPS 203 바이트 형식)를 보드 RAM에 적재.
    정적 키 모델: 한 번 올리고 같은 키로 decaps를 반복."""
    send_chunks("s", dk)                          # 's' 명령으로 dk 조각 전송


def decaps(ct, capture=False):
    """암호문을 보내고 보드에서 crypto_kem_dec 실행 -> 공유비밀 32B 반환.
    capture=True면 그 decaps 동안의 파형을 수집."""
    send_chunks("c", ct)                         # 1) 암호문 적재 ('c')
    if capture:                                  #    파형 수집 모드면
        scope.arm()                              # 2) 'd' 보내기 "전에" arm 해야 트리거를 안 놓침
    target.simpleserial_write("d", bytearray())  # 3) 복호화 시작 (clean+TRIGGER면 A/B 트리거 발생)
    if capture and scope.capture():              # 타임아웃이면 이전 파형을 잘못 저장하지 않도록 중단
        raise TimeoutError("B 트리거 캡처 타임아웃: 플래시/트리거/샘플 설정 확인")
    ss = target.simpleserial_read("r", 32, timeout=3000)  # 5) 공유비밀 32B 수신 (+ ack)
    return None if ss is None else bytes(ss)     # 응답 없으면 None


# ------------------------------------------------------------
# 정상 암호문 4회 + 변조 암호문 1회로 decaps 결과 비교
# ------------------------------------------------------------
ek, dk = KEM.keygen()   # PC에서 키 쌍 생성 (ek: 공개키, dk: 비밀키)
load_key(dk)            # 비밀키는 보드에 한 번만 적재 (정적 키)

for i in range(5):                               # 5회 반복
    key, ct = KEM.encaps(ek)          # 정상 암호문 + 정답 공유비밀(key)
    if i == 4:                        # 마지막 1회는 변조 암호문 테스트
        # 변조 암호문: 첫 바이트의 1비트 반전
        # FO 변환이라 복호화 -> 재암호화 -> 비교에서 불일치 -> implicit rejection
        # (에러 대신 dk 안의 z와 ct로 만든 가짜 키를 반환)
        ct = bytes([ct[0] ^ 1]) + ct[1:]         # 첫 바이트 LSB 뒤집기
        key = KEM.decaps(dk, ct)      # 정답도 PC에서 똑같이 rejection 값으로 계산
    ss = decaps(ct)                   # 보드 결과
    # 일치하면 정상/거부 경로 모두 FIPS 203과 같게 동작
    # 변조 암호문이어도 indcpa_dec는 그대로 돌기 때문에 A/B 트리거는 발생함
    print(i, "OK" if ss == key else f"MISMATCH {ss}")   # 보드 vs PC 비교

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:703) Target clock may drop; you may need to reset your target.


adc_mul=2  adc_freq=14.74 MHz  (clkgen 7.37MHz × adc_mul, target reset done)
firmware: 768 clean trigger=0b10 (bit0=A bit1=B) chunk 240
0 OK
1 OK
2 OK
3 OK
4 OK


In [ ]:
from tqdm.notebook import tnrange              # 진행바
import pathlib, time                               # 경로/타임스탬프
import numpy as np                                # 배열/저장

scope.adc.samples = SAMPLES                        # 캡처 길이 적용
scope.adc.stream_mode = SAMPLES > 131071           # 131071 초과 시에만 stream (2×+125000이면 False)
print(f"samples={scope.adc.samples}  stream_mode={scope.adc.stream_mode}")

out = pathlib.Path("../traces"); out.mkdir(exist_ok=True)   # 저장 폴더
traces, cts, keys = [], [], []                    # 파형/암호문/공유비밀 누적

for n in tnrange(N_TRACES):                       # N_TRACES번 수집
    key, ct = KEM.encaps(ek)                      # 매 반복 새 암호문(입력 변화) + 정답 공유비밀
    ss = decaps(ct, capture=True)                 # 보드 decaps + 파형 캡처
    if ss != key:                                 # 통신 오류 등으로 결과 불일치면
        continue                                  #   그 파형은 버림
    traces.append(scope.get_last_trace())         # 방금 캡처한 파형 저장
    cts.append(np.frombuffer(ct, dtype=np.uint8)) # 대응 암호문 (traces와 정렬)
    keys.append(np.frombuffer(key, dtype=np.uint8))  # 대응 공유비밀
    if n == 0:                                    # 첫 B 캡처에서 트리거 길이 점검
        print("trigger high samples:", scope.adc.trig_count, "/ 캡처", SAMPLES,
              "(trig_count > 캡처면 B 함수 뒷부분 잘림 -> SAMPLES 키우기)")

stamp = time.strftime("%Y%m%d-%H%M%S")            # 파일명용 타임스탬프
# 파일명에 스킴(PARAMS)·구현(IMPL)·최적화(OPT) 포함 -> 수집 조건 바로 식별
path = out / f"mlkem{PARAMS}-{IMPL}-O{OPT}-poly_tomsg-{stamp}.npz"
np.savez_compressed(path,
                    traces=np.array(traces, dtype=np.float32), ct=np.array(cts), ss=np.array(keys),  # 파형+ct+ss
                    dk=np.frombuffer(dk, dtype=np.uint8), ek=np.frombuffer(ek, dtype=np.uint8),      # 재현용 키
                    scheme=np.array(PARAMS), impl=np.array(IMPL), opt=np.array(OPT),                 # 메타: 512/768/1024, 구현, 최적화
                    samples=np.array(SAMPLES), adc_mul=np.array(ADC_MUL),
                    capture_target=np.array("poly_tomsg"))                            # 메타: 캡처 길이, ADC 배수
print(len(traces), "traces saved ->", path.name)  # 저장 개수 + 파일명

samples=20000  stream_mode=False


  0%|          | 0/1000 [00:00<?, ?it/s]

trigger high samples: 7170 / 캡처 20000 (trig_count > 캡처면 B 함수 뒷부분 잘림 -> SAMPLES 키우기)
